# Facial Emotion Classifier — Final Version

**Goal:** Classify facial expressions into 7 emotions using a CNN in PyTorch.

**Pipeline:** 48×48 grayscale → training augmentation → CNN → 7-class prediction → accuracy + precision/recall/F1 + confusion matrix.

**Important:** The original test set is kept completely untouched. The original training set is split into training and validation subsets for model selection.

In [ ]:
import os, random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

## 1. Load FER2013

Expected structure:

`fer2013/train/<emotion>/images`  
`fer2013/test/<emotion>/images`

The seven classes are angry, disgust, fear, happy, neutral, sad and surprise.

In [ ]:
# If using Colab and your dataset is a ZIP, uncomment:
# from google.colab import files
# import zipfile
# uploaded = files.upload()
# with zipfile.ZipFile("archive.zip", "r") as z:
#     z.extractall("fer2013")

print(os.listdir("fer2013"))

In [ ]:
train_aug_transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((48, 48)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

eval_transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

full_train_aug = datasets.ImageFolder("fer2013/train", transform=train_aug_transform)
full_train_eval = datasets.ImageFolder("fer2013/train", transform=eval_transform)
test_dataset = datasets.ImageFolder("fer2013/test", transform=eval_transform)

print("Classes:", full_train_aug.classes)
print("Full train:", len(full_train_aug))
print("Test:", len(test_dataset))

In [ ]:
# 90% train / 10% validation split
val_size = int(0.10 * len(full_train_aug))
train_size = len(full_train_aug) - val_size

g = torch.Generator().manual_seed(SEED)
train_subset, val_subset = random_split(range(len(full_train_aug)), [train_size, val_size], generator=g)

train_dataset = torch.utils.data.Subset(full_train_aug, train_subset.indices)
val_dataset = torch.utils.data.Subset(full_train_eval, val_subset.indices)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

imgs, labels = next(iter(train_loader))
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))
print("Batch shape:", imgs.shape)

### Preprocessing

- **Grayscale:** one input channel and matches the dataset format.
- **48×48:** compact image size.
- **RandomHorizontalFlip + RandomRotation:** training-only augmentation to improve generalization.
- **Normalization:** standardizes pixel values.
- Validation and test data receive **no random augmentation**.

## 2. CNN Architecture

In [ ]:
class EmotionCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 6 * 6, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 7)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = EmotionCNN().to(DEVICE)
print("Output shape:", model(torch.zeros(2,1,48,48).to(DEVICE)).shape)

**Architecture:** 48×48×1 → 32 filters → 64 → 128 → 6×6 feature maps → 512-unit layer → 7 emotion outputs.

MaxPool reduces spatial size; BatchNorm stabilizes training; Dropout helps reduce overfitting.

## 3. Training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_epoch(model, loader):
    model.train()
    loss_sum = correct = total = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)

    return loss_sum / total, correct / total

def evaluate(model, loader):
    model.eval()
    loss_sum = correct = total = 0

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, labels)

            loss_sum += loss.item() * imgs.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += imgs.size(0)

    return loss_sum / total, correct / total

In [ ]:
EPOCHS = 50
PATIENCE = 7
best_val_acc = 0
wait = 0

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        wait = 0
        torch.save(model.state_dict(), "best_model.pt")
        tag = " <- best"
    else:
        wait += 1
        tag = ""

    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"train loss {train_loss:.4f} acc {train_acc:.3f} | "
          f"val loss {val_loss:.4f} acc {val_acc:.3f}{tag}")

    if wait >= PATIENCE:
        print("Early stopping.")
        break

print(f"Best validation accuracy: {best_val_acc:.4f}")

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history["train_acc"], label="Train Accuracy")
plt.plot(history["val_acc"], label="Validation Accuracy")
plt.xlabel("Epoch"); plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.show()

plt.figure(figsize=(8,5))
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()

## 4. Final Test Evaluation

The best checkpoint is selected using the **validation set only**. The test set is evaluated once at the end, so it remains an unbiased final evaluation set.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

model.load_state_dict(torch.load("best_model.pt", map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        outputs = model(imgs.to(DEVICE))
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

test_acc = accuracy_score(all_labels, all_preds)
print(f"Final test accuracy: {test_acc:.4f}")
print()
print(classification_report(
    all_labels, all_preds,
    target_names=test_dataset.classes,
    digits=4
))

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(8,7))
disp = ConfusionMatrixDisplay(cm, display_labels=test_dataset.classes)
disp.plot(ax=ax, values_format="d", xticks_rotation=45)
plt.title("FER2013 Test Confusion Matrix")
plt.show()

### How to discuss the confusion matrix

It shows which emotions are being confused with one another. This gives more detail than accuracy alone and helps identify classes that need improvement.

## 5. Single Image Prediction

In [ ]:
from PIL import Image

class_names = test_dataset.classes

def predict_image(image_path):
    image = Image.open(image_path).convert("RGB")
    x = eval_transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1).squeeze().cpu().numpy()

    pred = probs.argmax()
    print(f"Predicted: {class_names[pred].upper()}")
    print(f"Confidence: {probs[pred]*100:.1f}%")

    for name, p in zip(class_names, probs):
        print(f"{name:10s}: {p*100:.1f}%")

    return class_names[pred], probs

## 6. Optional Streamlit Deployment

The trained `best_model.pt` can be used in the Streamlit application:

**Upload image → grayscale/resize/normalize → CNN → softmax → predicted emotion + confidence**

The deployment part is separate from model training.

In [ ]:
# Save the selected final checkpoint
torch.save(model.state_dict(), "best_model.pt")
print("Saved best_model.pt")

# Interview Cheat Sheet

**Why CNN?** CNNs learn spatial patterns in images such as edges, textures and facial features.

**Why grayscale?** The dataset is grayscale and one channel reduces computation.

**Why 32 → 64 → 128 filters?** Deeper layers learn richer features while spatial dimensions shrink.

**Why MaxPool?** Reduces spatial dimensions and computation.

**Why BatchNorm?** Helps stabilize and speed up training.

**Why Dropout?** Helps reduce overfitting.

**Why CrossEntropyLoss?** This is a 7-class classification problem.

**Why Adam?** It adapts the learning rate for model parameters and works well for neural networks.

**How did you prevent leakage?** I split the original training data into train and validation sets. I selected the best model using validation performance and kept the original test set untouched until final evaluation.

**Why augmentation?** Small flips and rotations create training variation and can improve generalization.

**How did you evaluate?** Accuracy, precision, recall, F1-score and a confusion matrix.

**What would you improve next?** More careful class-imbalance handling, hyperparameter tuning, stronger label-preserving augmentation and transfer learning.